# EDA — Freddie Mac Sample 2016

**Mục đích:** khám phá dữ liệu thô trước khi cứng hóa logic vào `scripts/01_build_trajectory.py`. Notebook này KHÔNG phải deliverable, không cần chạy lại từ đầu đến cuối (xem `PROJECT_BRIEF.md` §7bis).

**Câu hỏi cần trả lời:**
1. `Current Loan Delinquency Status` (perf, field 4) có những mã giá trị gì? Map sang 5 state `{Current, 30, 60, 90+, Default}` thế nào?
2. `Zero Balance Code` (perf, field 9) dùng để xác định chính xác trạng thái hấp thụ Default/Foreclosure vs trả hết (prepaid) — phân phối giá trị ra sao?
3. Mỗi loan có bao nhiêu tháng quan sát? Khoảng thời gian dữ liệu trải từ đâu đến đâu (để chia estimation/validation theo thời gian)?
4. `loan_id` giữa file origination và performance có khớp hết không?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path('..') / 'data' / 'raw'
ORIG_FILE = RAW_DIR / 'sample_orig_2016.txt'
PERF_FILE = RAW_DIR / 'sample_perf_2016.txt'

# smoke test: gioi han so dong doc tu file performance (3.38M dong, 366MB)
# doi thanh None de doc full khi da san sang
PERF_SAMPLE_ROWS = 500_000

## 1. File origination (1 dong / loan)

Column mapping theo layout chinh thuc Freddie Mac — ban Sample bo cot `Servicer Name` nen chi con 31 cot (thay vi 32 o ban Standard).

In [ ]:
ORIG_COLUMNS = [
    'credit_score', 'first_payment_date', 'first_time_homebuyer_flag', 'maturity_date',
    'msa', 'mi_pct', 'num_units', 'occupancy_status', 'cltv', 'dti', 'orig_upb',
    'orig_ltv', 'orig_interest_rate', 'channel', 'ppm_flag', 'amortization_type',
    'property_state', 'property_type', 'postal_code', 'loan_id', 'loan_purpose',
    'orig_loan_term', 'num_borrowers', 'seller_name', 'super_conforming_flag',
    'pre_harp_loan_seq_num', 'program_indicator', 'harp_indicator',
    'property_valuation_method', 'io_flag', 'mi_cancellation_indicator',
]

orig = pd.read_csv(ORIG_FILE, sep='|', header=None, names=ORIG_COLUMNS, dtype=str)
print(orig.shape)
orig.head()

In [ ]:
orig.isna().mean().sort_values(ascending=False)

In [ ]:
orig['loan_id'].duplicated().sum()  # ky vong 0 - loan_id phai unique trong orig file

## 2. File performance (nhieu dong / loan, 1 dong / thang)

Chi 9 cot dau la can chac chan cho project nay (loan_id, thang, trang thai DPD, zero balance code). Cac cot 10-35 la chi tiet loss/modification khong dung den — de ten generic `colNN`, khong doan bua.

In [ ]:
PERF_COLUMNS = [
    'loan_id', 'monthly_reporting_period', 'current_upb', 'delinquency_status',
    'loan_age', 'remaining_months', 'defect_settlement_date', 'modification_flag',
    'zero_balance_code', 'zero_balance_date',
] + [f'col{i}' for i in range(11, 36)]

perf = pd.read_csv(
    PERF_FILE, sep='|', header=None, names=PERF_COLUMNS, dtype=str,
    nrows=PERF_SAMPLE_ROWS,
)
print(perf.shape)
perf.head()

### 2.1 Phan phoi Current Loan Delinquency Status — co so de rori rac hoa 5 state

In [ ]:
perf['delinquency_status'].value_counts(dropna=False).sort_index()

### 2.2 Phan phoi Zero Balance Code — xac dinh trang thai hap thu (Default/Foreclosure vs tra het)

In [ ]:
perf['zero_balance_code'].value_counts(dropna=False)

### 2.3 So thang quan sat / loan, khoang thoi gian

In [ ]:
months_per_loan = perf.groupby('loan_id')['monthly_reporting_period'].count()
print(months_per_loan.describe())
print('Khoang thoi gian:', perf['monthly_reporting_period'].min(), '->', perf['monthly_reporting_period'].max())

### 2.4 Khop loan_id giua orig va perf

In [ ]:
orig_ids = set(orig['loan_id'])
perf_ids = set(perf['loan_id'])
print('So loan trong orig:', len(orig_ids))
print('So loan trong perf (sample doc):', len(perf_ids))
print('perf_ids khong co trong orig_ids:', len(perf_ids - orig_ids))

## 3. Ghi chu / ket luan

_(dien sau khi chay xong cac cell tren — mapping DPD status -> 5 state, zero balance code nao tinh la Default/Foreclosure, co can loc bo loan nao khong)_